In [15]:
import importlib
import pprint

import ml_filter as mf
importlib.reload(mf)

<module 'ml_filter' from 'c:\\Users\\dwack\\Documents\\Python Scripts\\sh_ga\\ga_sh\\ml_filter.py'>

In [16]:
bulk_model, surf_model = mf.load_models(force_reload=True)

print("Bulk model loaded:", type(bulk_model))
print("Surf model loaded:", type(surf_model))

Bulk model loaded: <class 'sklearn.ensemble._forest.RandomForestRegressor'>
Surf model loaded: <class 'sklearn.ensemble._forest.RandomForestRegressor'>


In [17]:
print("FEATURE_COLS:")
print(mf.FEATURE_COLS)
print()

print("DOMAIN_BOUNDS:")
pprint.pprint(mf.DOMAIN_BOUNDS)
print()

print("DEFAULT_THRESHOLDS:")
pprint.pprint(mf.DEFAULT_THRESHOLDS)

FEATURE_COLS:
['Mn', 'Si', 'Cr', 'Mo', 'Ni', 'Cu', 'Co', 'V', 'Nb', 'W', 'Al', 'Ti']

DOMAIN_BOUNDS:
{'Cr': (8.0, 20.0),
 'Cu': (0.0, 4.0),
 'Mn': (0.0, 3.0),
 'Mo': (0.0, 6.0),
 'Nb': (0.0, 4.0),
 'Ni': (0.0, 14.0),
 'Si': (0.0, 1.5),
 'Ti': (0.0, 4.0),
 'W': (0.0, 5.0)}

DEFAULT_THRESHOLDS:
{'bulk_positive_margin': 0.0,
 'bulk_uncertainty_band': 0.1,
 'compare_margin': 0.0,
 'compare_uncertainty_band': 0.17,
 'surf_nonpositive_margin': 0.0,
 'surf_uncertainty_band': 0.14}


In [18]:
def show_ml_result(title, result):
    print("=" * 80)
    print(title)
    print("=" * 80)
    pprint.pprint(result)
    print()
    print("should_continue_after_ml =", mf.should_continue_after_ml(result))
    print()

In [19]:
base_comp = {
    "Cr": 16.5,
    "Mn": 0.4,
    "Si": 0.4,
    "Mo": 2.0,
    "Nb": 0.4,
    "Ti": 0.2,
    "Ni": 2.0,
    "Cu": 0.2,
    "C": 0.05,
    "Fe": 77.85,   # no lo usa el modelo, pero puede venir del encoder
}

feat = mf.build_feature_dict(base_comp)
print("FEATURE DICT:")
pprint.pprint(feat)

X = mf.build_feature_frame(base_comp)
print("\nFEATURE FRAME:")
display(X)

domain = mf.check_domain(feat)
print("\nDOMAIN CHECK:")
pprint.pprint(domain)

preds = mf.predict_df_ml(base_comp, bulk_model=bulk_model, surf_model=surf_model)
print("\nPREDICTIONS:")
pprint.pprint(preds)

FEATURE DICT:
{'Al': 0.0,
 'Co': 0.0,
 'Cr': 16.5,
 'Cu': 0.2,
 'Mn': 0.4,
 'Mo': 2.0,
 'Nb': 0.4,
 'Ni': 2.0,
 'Si': 0.4,
 'Ti': 0.2,
 'V': 0.0,
 'W': 0.0}

FEATURE FRAME:


,Mn,Si,Cr,Mo,Ni,Cu,Co,V,Nb,W,Al,Ti
0,0.4,0.4,16.5,2.0,2.0,0.2,0.0,0.0,0.4,0.0,0.0,0.2



DOMAIN CHECK:
{'in_domain': True, 'out_of_domain_features': {}}

PREDICTIONS:
{'DF_bulk_ml': 0.482115, 'DF_surf_ml': 0.407655}


In [20]:
comp_bulk_dom = {
    "Cr": 17.0,
    "Mn": 0.5,
    "Si": 0.3,
    "Mo": 2.5,
    "Nb": 0.2,
    "Ti": 0.1,
    "Ni": 1.0,
    "Cu": 0.0,
    "C": 0.05,
    "Fe": 78.35,
}

result_bulk_dom = mf.evaluate_ml_filter(
    composition_bulk=comp_bulk_dom,
    mode="bulk_dominant",
    bulk_model=bulk_model,
    surf_model=surf_model,
)

show_ml_result("CASE 1 — bulk_dominant candidate", result_bulk_dom)

CASE 1 — bulk_dominant candidate
{'DF_bulk_ml': 0.334233,
 'DF_surf_ml': 0.371972,
 'feature_vector_ml': {'Al': 0.0,
                       'Co': 0.0,
                       'Cr': 17.0,
                       'Cu': 0.0,
                       'Mn': 0.5,
                       'Mo': 2.5,
                       'Nb': 0.2,
                       'Ni': 1.0,
                       'Si': 0.3,
                       'Ti': 0.1,
                       'V': 0.0,
                       'W': 0.0},
 'in_domain': True,
 'ml_status': 'reject',
 'mode': 'bulk_dominant',
 'out_of_domain_features': {},
 'reason': 'surface_effect_present'}

should_continue_after_ml = False



In [21]:
comp_surface_enh = {
    "Cr": 15.0,
    "Mn": 0.5,
    "Si": 0.8,
    "Mo": 1.5,
    "Nb": 1.0,
    "Ti": 0.8,
    "Ni": 3.0,
    "Cu": 0.5,
    "C": 0.05,
    "Fe": 77.85,
}

result_surface_enh = mf.evaluate_ml_filter(
    composition_bulk=comp_surface_enh,
    mode="surface_enhanced",
    bulk_model=bulk_model,
    surf_model=surf_model,
)

show_ml_result("CASE 2 — surface_enhanced candidate", result_surface_enh)

CASE 2 — surface_enhanced candidate
{'DF_bulk_ml': 0.460147,
 'DF_surf_ml': 0.416083,
 'feature_vector_ml': {'Al': 0.0,
                       'Co': 0.0,
                       'Cr': 15.0,
                       'Cu': 0.5,
                       'Mn': 0.5,
                       'Mo': 1.5,
                       'Nb': 1.0,
                       'Ni': 3.0,
                       'Si': 0.8,
                       'Ti': 0.8,
                       'V': 0.0,
                       'W': 0.0},
 'in_domain': True,
 'ml_status': 'uncertain_model',
 'mode': 'surface_enhanced',
 'out_of_domain_features': {},
 'reason': 'surface_vs_bulk_near_boundary'}

should_continue_after_ml = True



In [22]:
thresholds_wide = {
    "bulk_positive_margin": 0.0,
    "surf_nonpositive_margin": 0.0,
    "compare_margin": 0.0,
    "bulk_uncertainty_band": 0.30,
    "surf_uncertainty_band": 0.30,
    "compare_uncertainty_band": 0.30,
}

comp_uncertain_model = {
    "Cr": 16.0,
    "Mn": 0.6,
    "Si": 0.5,
    "Mo": 1.5,
    "Nb": 0.5,
    "Ti": 0.3,
    "Ni": 2.0,
    "Cu": 0.2,
    "C": 0.05,
    "Fe": 78.35,
}

result_uncertain_model = mf.evaluate_ml_filter(
    composition_bulk=comp_uncertain_model,
    mode="surface_enhanced",
    thresholds=thresholds_wide,
    bulk_model=bulk_model,
    surf_model=surf_model,
)

show_ml_result("CASE 3 — uncertain_model candidate", result_uncertain_model)

CASE 3 — uncertain_model candidate
{'DF_bulk_ml': 0.445816,
 'DF_surf_ml': 0.410435,
 'feature_vector_ml': {'Al': 0.0,
                       'Co': 0.0,
                       'Cr': 16.0,
                       'Cu': 0.2,
                       'Mn': 0.6,
                       'Mo': 1.5,
                       'Nb': 0.5,
                       'Ni': 2.0,
                       'Si': 0.5,
                       'Ti': 0.3,
                       'V': 0.0,
                       'W': 0.0},
 'in_domain': True,
 'ml_status': 'uncertain_model',
 'mode': 'surface_enhanced',
 'out_of_domain_features': {},
 'reason': 'surface_vs_bulk_near_boundary'}

should_continue_after_ml = True



In [23]:
comp_outside_domain = {
    "Cr": 19.0,
    "Mn": 4.5,   # outside domain (Mn > 3.0)
    "Si": 1.8,   # outside domain (Si > 1.5)
    "Mo": 5.5,
    "Nb": 2.0,
    "Ti": 2.0,
    "Ni": 10.0,
    "Cu": 0.5,
    "C": 0.05,
    "Fe": 54.65,
}

result_uncertain_domain = mf.evaluate_ml_filter(
    composition_bulk=comp_outside_domain,
    mode="bulk_dominant",
    bulk_model=bulk_model,
    surf_model=surf_model,
)

show_ml_result("CASE 4 — uncertain_domain candidate", result_uncertain_domain)

CASE 4 — uncertain_domain candidate
{'DF_bulk_ml': 0.385772,
 'DF_surf_ml': 0.183013,
 'feature_vector_ml': {'Al': 0.0,
                       'Co': 0.0,
                       'Cr': 19.0,
                       'Cu': 0.5,
                       'Mn': 4.5,
                       'Mo': 5.5,
                       'Nb': 2.0,
                       'Ni': 10.0,
                       'Si': 1.8,
                       'Ti': 2.0,
                       'V': 0.0,
                       'W': 0.0},
 'in_domain': False,
 'ml_status': 'uncertain_domain',
 'mode': 'bulk_dominant',
 'out_of_domain_features': {'Mn': {'hi': 3.0, 'lo': 0.0, 'value': 4.5},
                            'Si': {'hi': 1.5, 'lo': 0.0, 'value': 1.8}},
 'reason': 'outside_training_domain'}

should_continue_after_ml = True



In [24]:
test_compositions = [
    {"Cr": 17.0, "Mn": 0.5, "Si": 0.3, "Mo": 2.5, "Nb": 0.2, "Ti": 0.1, "Ni": 1.0, "Cu": 0.0, "C": 0.05},
    {"Cr": 15.0, "Mn": 0.5, "Si": 0.8, "Mo": 1.5, "Nb": 1.0, "Ti": 0.8, "Ni": 3.0, "Cu": 0.5, "C": 0.05},
    {"Cr": 18.0, "Mn": 0.2, "Si": 0.2, "Mo": 3.0, "Nb": 0.0, "Ti": 0.0, "Ni": 0.5, "Cu": 0.0, "C": 0.05},
    {"Cr": 14.5, "Mn": 1.0, "Si": 1.0, "Mo": 0.8, "Nb": 1.5, "Ti": 1.0, "Ni": 4.0, "Cu": 1.0, "C": 0.05},
    {"Cr": 16.0, "Mn": 0.8, "Si": 0.6, "Mo": 2.0, "Nb": 0.5, "Ti": 0.2, "Ni": 2.5, "Cu": 0.2, "C": 0.05},
]

for i, comp in enumerate(test_compositions, start=1):
    res_bulk = mf.evaluate_ml_filter(
        composition_bulk=comp,
        mode="bulk_dominant",
        bulk_model=bulk_model,
        surf_model=surf_model,
    )
    res_surf = mf.evaluate_ml_filter(
        composition_bulk=comp,
        mode="surface_enhanced",
        bulk_model=bulk_model,
        surf_model=surf_model,
    )

    print("=" * 100)
    print(f"COMPOSITION {i}")
    pprint.pprint(comp)
    print("\nBULK_DOMINANT:")
    pprint.pprint({
        "ml_status": res_bulk["ml_status"],
        "reason": res_bulk["reason"],
        "DF_bulk_ml": res_bulk["DF_bulk_ml"],
        "DF_surf_ml": res_bulk["DF_surf_ml"],
    })
    print("\nSURFACE_ENHANCED:")
    pprint.pprint({
        "ml_status": res_surf["ml_status"],
        "reason": res_surf["reason"],
        "DF_bulk_ml": res_surf["DF_bulk_ml"],
        "DF_surf_ml": res_surf["DF_surf_ml"],
    })
    print()

COMPOSITION 1
{'C': 0.05,
 'Cr': 17.0,
 'Cu': 0.0,
 'Mn': 0.5,
 'Mo': 2.5,
 'Nb': 0.2,
 'Ni': 1.0,
 'Si': 0.3,
 'Ti': 0.1}

BULK_DOMINANT:
{'DF_bulk_ml': 0.334233,
 'DF_surf_ml': 0.371972,
 'ml_status': 'reject',
 'reason': 'surface_effect_present'}

SURFACE_ENHANCED:
{'DF_bulk_ml': 0.334233,
 'DF_surf_ml': 0.371972,
 'ml_status': 'uncertain_model',
 'reason': 'surface_vs_bulk_near_boundary'}

COMPOSITION 2
{'C': 0.05,
 'Cr': 15.0,
 'Cu': 0.5,
 'Mn': 0.5,
 'Mo': 1.5,
 'Nb': 1.0,
 'Ni': 3.0,
 'Si': 0.8,
 'Ti': 0.8}

BULK_DOMINANT:
{'DF_bulk_ml': 0.460147,
 'DF_surf_ml': 0.416083,
 'ml_status': 'reject',
 'reason': 'surface_effect_present'}

SURFACE_ENHANCED:
{'DF_bulk_ml': 0.460147,
 'DF_surf_ml': 0.416083,
 'ml_status': 'uncertain_model',
 'reason': 'surface_vs_bulk_near_boundary'}

COMPOSITION 3
{'C': 0.05,
 'Cr': 18.0,
 'Cu': 0.0,
 'Mn': 0.2,
 'Mo': 3.0,
 'Nb': 0.0,
 'Ni': 0.5,
 'Si': 0.2,
 'Ti': 0.0}

BULK_DOMINANT:
{'DF_bulk_ml': 0.219935,
 'DF_surf_ml': 0.229085,
 'ml_status': 're

In [25]:
for title, result in [
    ("bulk_dominant", result_bulk_dom),
    ("surface_enhanced", result_surface_enh),
    ("uncertain_model", result_uncertain_model),
    ("uncertain_domain", result_uncertain_domain),
]:
    print(title, "->", result["ml_status"], "-> continue?", mf.should_continue_after_ml(result))

bulk_dominant -> reject -> continue? False
surface_enhanced -> uncertain_model -> continue? True
uncertain_model -> uncertain_model -> continue? True
uncertain_domain -> uncertain_domain -> continue? True
